# Reinforcement learning algorithms primer

## Environment model (grid world)

In [19]:
from random import random
import numpy as np

# Board as in example:
#https://www.davidsilver.uk/wp-content/uploads/2020/03/DP.pdf

BOARD_ROWS = 4
BOARD_COLS = 4
WIN_STATEs = [(0, 0), (3, 3)]
LOSE_STATEs = []
START = (2, 0)
DETERMINISTIC = True

class Environment:
    def __init__(self, state=START):

        self.actions = ["up", "down", "left", "right"]
        self.board = np.zeros([BOARD_ROWS, BOARD_COLS])
        self.board[1, 1] = -1
        self.state = state
        self.isEnd = False
        self.determine = DETERMINISTIC

        self.strategy= np.zeros([BOARD_ROWS, BOARD_COLS, len(self.actions)])+0.25

        self.state_values = {}
        # Snapshots of state_values after every iteration used by the plotting helpers
        self.history = []
        self.reset_state_values_to_unknown()


    def get_random_nonterminal_state(self):
        terminal_state = True
        state = None
        while terminal_state:
            state = (np.random.randint(BOARD_ROWS),  np.random.randint(BOARD_COLS))
            terminal_state = self.isTerminalState(state)
        return state


    def zeros_state_values(self):
        for i in range(BOARD_ROWS):
            for j in range(BOARD_COLS):
                self.state_values[(i, j)] = 0


    def reset_state_values_to_unknown(self):
        for i in range(BOARD_ROWS):
            for j in range(BOARD_COLS):
                if self.isTerminalState((i,j)):
                    self.state_values[(i, j)] = 0
                else:
                    self.state_values[(i, j)] = -100000


    def getNextState(self, from_state, action):

        if self.isTerminalState(from_state):
            return from_state

        if action == "up":
            nxtState = (from_state[0] - 1, from_state[1])
        elif action == "down":
            nxtState = (from_state[0] + 1, from_state[1])
        elif action == "left":
            nxtState = (from_state[0], from_state[1] - 1)
        else:
            nxtState = (from_state[0], from_state[1] + 1)
        # if next state legal
        if (nxtState[0] >= 0) and (nxtState[0] <= (BOARD_ROWS - 1)):
            if (nxtState[1] >= 0) and (nxtState[1] <= (BOARD_COLS - 1)):
                #if nxtState != (1, 1):
                return nxtState

        return from_state


    def isTerminalState(self,state):
        if (state in(WIN_STATEs)) or (state in(LOSE_STATEs)):
            return True
        return False


    def nxtPosition(self, action):
        """
        action: up, down, left, right
        -------------
        0 | 1 | 2| 3|
        1 |
        2 |
        return next position
        """
        if self.determine:
            if action == "up":
                nxtState = (self.state[0] - 1, self.state[1])
            elif action == "down":
                nxtState = (self.state[0] + 1, self.state[1])
            elif action == "left":
                nxtState = (self.state[0], self.state[1] - 1)
            else:
                nxtState = (self.state[0], self.state[1] + 1)

        else:
        # non-deterministic
            action = self._chooseActionProb(action)
            nxtState = self.nxtPosition(action)

        # if next state legal
        if (nxtState[0] >= 0) and (nxtState[0] <= (BOARD_ROWS - 1)):
            if (nxtState[1] >= 0) and (nxtState[1] <= (BOARD_COLS - 1)):
                #if nxtState != (1, 1):
                return nxtState

        return (self.state)



    def nextStateReward(self, current_state, action):
        """
        action: up, down, left, right
        -------------
        0 | 1 | 2| 3|
        1 |
        2 |
        return next position
        """

        reward = self.getReward(current_state, action)
        nxtState = self.getNextState(current_state, action)

        return [nxtState, reward]


    def _chooseActionProb(self, action):
        if action == "up":
            return np.random.choice(["up", "left", "right"], p=[0.8, 0.1, 0.1])
        if action == "down":
            return np.random.choice(["down", "left", "right"], p=[0.8, 0.1, 0.1])
        if action == "left":
            return np.random.choice(["left", "up", "down"], p=[0.8, 0.1, 0.1])
        if action == "right":
            return np.random.choice(["right", "up", "down"], p=[0.8, 0.1, 0.1])


    def showStateValues(self):
        print('---------STATE VALUES------------')
        for i in range(0, BOARD_ROWS):
            print('----------------------------------')
            out = '| '
            for j in range(0, BOARD_COLS):
                if True:
                    #"{0:0.2f}".format(self.state_values[(i, j)])
                    out += "{0:0.2f}".format(self.state_values[(i, j)]).ljust(6) + ' | '
            print(out)
        print('----------------------------------')


    def showStrategy(self):
        print('---------STRATEGY------------')
        for i in range(0, BOARD_ROWS):
            print('----------------------------------------------------------------------')
            out = '| '
            for j in range(0, BOARD_COLS):
                if True:
                    out += str(self.strategy[i, j]).ljust(6) + ' | '
            print(out)
        print('---------------------------------------------------------------------------')

    def random_policy(self, state):
        return [1.0/len(self.actions) for action in self.actions]


    def upright_policy(self, state):

        return [0.49,0.01,0.01,0.49]


    def apply_strategy(self, state):
        return self.strategy[state[0], state[1]]
    ### -------------------------------------------------------------------------------------------
    ### Implementation:
    ### -------------------------------------------------------------------------------------------


    ## REWARD MODEL R(s,a)
    def getReward(self, state, action):
        # The reward does NOT depend on the action or on where the agent lands (bumping into a wall is still a wasted step = -1)
        if self.isTerminalState(state):
            return 0
        return -1

    
    # VALUE ITERATION V(s) <- max_a [ r(s, a) + gamma * V(s') ]
    def DP(self, gamma=1):
        iterations = 0

        # history[0] = values BEFORE the first iteration 
        self.history = [dict(self.state_values)]

        while True:
            iterations += 1

            new_values = {}

            for state in self.state_values:
                # Terminal states always keep V = 0 episode is over
                if self.isTerminalState(state):
                    new_values[state] = 0
                    continue
                #   q = immediate reward + discounted value of the next state
                q_values = []
                for action in self.actions:
                    nxt_state, reward = self.nextStateReward(state, action)
                    q_values.append(reward + gamma * self.state_values[nxt_state])
                # value of a state = value of its best action
                new_values[state] = max(q_values)

            if new_values == self.state_values:
                break

            # Swap in the new values and record a snapshot for the plots
            self.state_values = new_values
            self.history.append(dict(self.state_values))

        return iterations


    # ITERATIVE POLICY EVALUATION V(s) <- sum_a pi(a|s) * [ r(s, a) + gamma * V(s') ]
    def value_iteration(self, epsilon, gamma=1):
        '''
        :param policy:
        :param epsilon: stop when the biggest change in one sweep < epsilon
        :param gamma:   discount factor
        :return:        None (prints "Iterations N")
        '''
        iterations = 0
        self.history = [dict(self.state_values)]

        while True:
            iterations += 1

            # delta = biggest |V_new(s) - V_old(s)| Must be reset every iteration
            delta = 0

            for state in self.state_values:

                # Terminal states stay at 0 
                if self.isTerminalState(state):
                    continue

                old_value = self.state_values[state]

                # pi(.|s): vector of 4 probabilities in the order of self.actions random policy -> [0.25, 0.25, 0.25, 0.25]
                probs = self.apply_strategy(state)

                # Expected value over actions under the policy sum_a pi(a|s) * (r + gamma * V(s'))
                new_value = 0
                for prob, action in zip(probs, self.actions):
                    nxt_state, reward = self.nextStateReward(state, action)
                    new_value += prob * (reward + gamma * self.state_values[nxt_state])

                # IN-PLACE update: write immediately
                self.state_values[state] = new_value

                # Track the largest change in this sweep
                delta = max(delta, abs(new_value - old_value))

            # Snapshot for the animation.
            self.history.append(dict(self.state_values))

            if delta < epsilon:
                break

        print("Iterations", iterations)


    # GREEDY POLICY IMPROVEMENT pi(s) <- argmax_a [ r(s, a) + gamma * V(s') ]
    def greedily_update_policy(self, epsilon, gamma=1):
        for (i, j) in self.state_values:

            # q(s, a) for all 4 actions - same formula as in DP() but instead of taking max(q) we remember best actions
            q_values = np.zeros(len(self.actions))
            for k, action in enumerate(self.actions):
                nxt_state, reward = self.nextStateReward((i, j), action)
                q_values[k] = reward + gamma * self.state_values[nxt_state]

            # Boolean mask  q = [-15, -21, -15, -21] -> [True, False, True, False]
            best = np.abs(q_values - q_values.max()) < epsilon

            # True/False -> probabilities that sum to 1 [True, False, True, False] / 2 -> [0.5, 0.0, 0.5, 0.0]
            self.strategy[i, j] = best / best.sum()
    
    
    

## Grid visualization

- `values_to_array` — converts the `state_values` dict into a 4×4 matrix

- `animate_values` — animated V(s) heatmap with an iteration slider (requires `env.history`)

- `policy_trace` — V(s) heatmap with policy arrows π(a|s) in every cell

- `compare_policies` — several policies side by side with a shared color scale

In [20]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ARROW_CHARS = {"up": "↑", "down": "↓", "left": "←", "right": "→"}


def values_to_array(state_values):
    """Dict {(i, j): V} -> numpy matrix of shape (BOARD_ROWS, BOARD_COLS)."""
    V = np.zeros((BOARD_ROWS, BOARD_COLS))
    for (i, j), v in state_values.items():
        V[i, j] = v
    return V


def animate_values(history, title, zmin):
    """Animated V(s) heatmap: one frame = one iteration of the algorithm."""
    frames = np.stack([values_to_array(h) for h in history])   # (T, 4, 4)
    frames = np.where(frames <= -100000, np.nan, frames)        # "unknown" -> empty cell
    fig = px.imshow(
        frames,
        animation_frame=0,                 # axis 0 = iteration -> slider + Play button
        text_auto=".1f",                   # values printed inside cells
        zmin=zmin, zmax=0,
        color_continuous_scale="RdYlGn",
        template="plotly_dark",
        labels=dict(animation_frame="iteration", color="V(s)", x="column", y="row"),
        title=title,
    )
    fig.update_traces(xgap=3, ygap=3)                       # gaps between cells = "tiles"
    fig.update_xaxes(dtick=1, showgrid=False, zeroline=False)
    fig.update_yaxes(dtick=1, showgrid=False, zeroline=False)
    fig.update_layout(width=520, height=560, font_size=14,
                      plot_bgcolor="#2b2b2b")                 # color of NaN ("unknown") cells
    ms = max(30, 4000 // len(history))                       # whole animation takes ~4 s
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = ms
    return fig


def policy_trace(env):
    """V(s) heatmap + action arrows from env.strategy (snapshot of the current state)."""
    V = values_to_array(env.state_values)
    text = []
    for i in range(BOARD_ROWS):
        row = []
        for j in range(BOARD_COLS):
            if env.isTerminalState((i, j)):
                row.append("<b>GOAL</b>")
                continue
            arrows = "".join(ARROW_CHARS[a] for k, a in enumerate(env.actions)
                             if env.strategy[i, j, k] > 0)
            row.append(f"{V[i, j]:.1f}<br><b>{arrows}</b>")
        text.append(row)
    return go.Heatmap(
        z=V, text=text, texttemplate="%{text}", textfont_size=15,
        customdata=env.strategy,           # (4, 4, 4) -> π(a|s) shown on hover
        hovertemplate="(%{y}, %{x})  V=%{z:.2f}<br>"
                      "π: up=%{customdata[0]:.2f} down=%{customdata[1]:.2f} "
                      "left=%{customdata[2]:.2f} right=%{customdata[3]:.2f}<extra></extra>",
        xgap=3, ygap=3, coloraxis="coloraxis",
    )


def compare_policies(traces, titles):
    """Several policy_trace snapshots side by side with a shared color scale."""
    fig = make_subplots(rows=1, cols=len(traces), subplot_titles=titles, horizontal_spacing=0.08)
    for k, tr in enumerate(traces, start=1):
        fig.add_trace(tr, row=1, col=k)
    fig.update_yaxes(autorange="reversed")        # row 0 at the top, same as in the code
    fig.update_xaxes(dtick=1, showgrid=False, zeroline=False)
    fig.update_yaxes(dtick=1, showgrid=False, zeroline=False)
    fig.update_layout(template="plotly_dark", width=900, height=460,
                      coloraxis=dict(colorscale="RdYlGn", cmax=0))
    return fig


print("Visualization helpers ready.")

Visualization helpers ready.


## Dynamic Programming

Value Iteration

In [21]:
print("----------------DP  - Value Iteration ----------------- ")
env = Environment()
env.reset_state_values_to_unknown()
env.DP()
env.showStateValues()
print("------------------------------------------------------- ")

----------------DP  - Value Iteration ----------------- 
---------STATE VALUES------------
----------------------------------
| 0.00   | -1.00  | -2.00  | -3.00  | 
----------------------------------
| -1.00  | -2.00  | -3.00  | -2.00  | 
----------------------------------
| -2.00  | -3.00  | -2.00  | -1.00  | 
----------------------------------
| -3.00  | -2.00  | -1.00  | 0.00   | 
----------------------------------
------------------------------------------------------- 


# Animation: Dynamic Programing Value Iteration:
* Gray cells = still "unknown" (-100000). After 3 iterations V* is reached

In [22]:
animate_values(env.history, "DP() — Value Iteration", zmin=-3).show()

## Iterative strategy evaluation
General Policy Improvement

In [23]:
print("----------------iterative strategy evaluation ----------------- ")
print("-----------env.iterative_strategy_evaluation( 0.00001, 1)-------")
env = Environment()
env.value_iteration( 0.00001, 1)
env.showStrategy()
env.showStateValues()
print("--------------------------------------------------------------- ")

history_random = list(env.history)   
t_random = policy_trace(env)        

print("----------------policy iteration  ----------------- ")
print("-----------env.greedily_update_policy(0.01, 1)-------")
env.greedily_update_policy(0.01, 1)
env.value_iteration( 0.00001, 1 )
env.showStateValues()
print("--------------------------------------------------------------- ")

----------------iterative strategy evaluation ----------------- 
-----------env.iterative_strategy_evaluation( 0.00001, 1)-------
Iterations 239
---------STRATEGY------------
----------------------------------------------------------------------
| [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | 
----------------------------------------------------------------------
| [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | 
----------------------------------------------------------------------
| [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | 
----------------------------------------------------------------------
| [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | [0.25 0.25 0.25 0.25] | 
---------------------------------------------------------------------------
---------STATE VALUES------------
----------------------------------


# Animation of the evaluation of the random policy 

In [24]:
animate_values(history_random, "Random policy evaluation", zmin=-22).show()

In [25]:
compare_policies([t_random, policy_trace(env)], ["Random π (V_random)", "Greedy π (V*)"]).show()